### Loading and Cleaning the Data

- Importing libraries and dependencies

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

- Loading Datasets

In [6]:
# Load all datasets
orders_df = pd.read_csv('../data/raw/orders.csv')
order_products_prior_df = pd.read_csv('../data/raw/order_products__prior.csv')
order_products_train_df = pd.read_csv('../data/raw/order_products__train.csv')
products_df = pd.read_csv('../data/raw/products.csv')
aisles_df = pd.read_csv('../data/raw/aisles.csv')
departments_df = pd.read_csv('../data/raw/departments.csv')

# Pandas Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


- Quick look at the Datasets

In [8]:
datasets = {
    'orders': orders_df,
    'order_products_prior': order_products_prior_df,
    'order_products_train': order_products_train_df,
    'products': products_df,
    'aisles': aisles_df
}

for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(df.dtypes)
    print(f"Nulls:\n{df.isnull().sum()}")


orders: 3,421,083 rows x 7 cols
order_id                    int64
user_id                     int64
eval_set                   object
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object
Nulls:
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

order_products_prior: 32,434,489 rows x 4 cols
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object
Nulls:
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

order_products_train: 1,384,617 rows x 4 cols
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object
Nulls:
order_

- Handling missing values: 

We have 206209 missing days_since_prior_order values in the orders dataframe, we can assume they are empty since the customer hasn't had any previous orders or we don't have data from any previous orders. So I will replace these values with 0. 

In [10]:
# Fill nulls in days_since_prior_order 
orders_df['days_since_prior_order'] = orders_df['days_since_prior_order'].fillna(0)

- How the data is divided

The Dataset has already been divided to prior, which is every customers prior order data and training which would be there most recent order. So using the prior data we will be trying to predict their most recent order.

In [13]:
print(f"\nUnique users: {orders_df['user_id'].nunique():,}")
print(f"Unique orders: {orders_df['order_id'].nunique():,}")
print(f"\nEval set distribution:\n{orders_df['eval_set'].value_counts()}")


Unique users: 206,209
Unique orders: 3,421,083

Eval set distribution:
eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64


- Merging all dfs together

In [14]:
# Merge order_products_prior with orders
prior = order_products_prior_df.merge(orders_df, on='order_id', how='left')

# Merge with product names, aisles, departments
prior = prior.merge(products_df, on='product_id', how='left')
prior = prior.merge(aisles_df, on='aisle_id', how='left')
prior = prior.merge(departments_df, on='department_id', how='left')

print(f"Master dataframe shape: {prior.shape}")
print(f"\nColumns: {prior.columns.tolist()}")
prior.head()

Master dataframe shape: (32434489, 15)

Columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_name', 'aisle_id', 'department_id', 'aisle', 'department']


,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.00,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.00,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.00,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.00,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.00,Natural Sweetener,17,13,baking ingredients,pantry
